# Laplacian Prior with Laplacian Likelihood: Exact Marginal by Transform Inversion

This notebook implements Section 6 of the draft, the `LF (exact)` row of Table 2 and the last
block of the filter hierarchy. The prior on $\theta_t$ becomes Laplacian, so the interference
$S_m = \sum_{n\neq m} x_{t,n}\theta_{t,n}$ leaves the Gaussian family and the closed-form
partial-fraction mixture (77) is unusable: its weights alternate in sign and grow without bound
as any two scales approach each other. What remains is the transform.

The model is fully Laplacian,

$$ \xi_t \sim \mathrm{Lap}(0, b_\xi), \qquad \eta_t \sim \mathrm{Lap}(0, b_\eta), \qquad b_\xi = \sqrt{\varepsilon/2}. $$

The convolution of two Laplacians is not Laplacian, so the predicted prior (75) is carried as
the convolution itself, and each predicted marginal contributes two elementary factors of scales
$c_n = b_{t-1}|x_{t,n}|$ and $g_n = b_\xi|x_{t,n}|$:

$$ \hat q_n(\omega) = \frac{1}{(1+c_n^2\omega^2)(1+g_n^2\omega^2)}, \qquad
   \Psi_t(\omega) = \frac{1}{1+b_\eta^2\omega^2}\prod_{n=1}^{M}\hat q_n(\omega), \qquad
   \hat f_{\zeta_m}(\omega) = \frac{\Psi_t(\omega)}{\hat q_m(\omega)}. $$

One product $\Psi_t$ serves all $M$ coefficients, which is what makes the step $O(M)$. Every
statistic the projection asks for is then a value of an inverse transform at the single point
$e_t = y_t - \boldsymbol{x}_t^{\mathsf T}\boldsymbol{w}_{t-1}$, so they all come from one functional

$$ \mathcal{I}_t[R] = \frac{1}{2\pi}\int_{\mathbb R} \Psi_t(\omega)\,R(\omega)\,e^{i\omega e_t}\,d\omega,
   \qquad Z_t = \mathcal{I}_t[1]. $$

The Laplacian family projects on the median and the MAD, not on the mean and the variance
(Remark 10: matching moments would project onto the Gaussian family and the Laplacian prior would
survive a single step). So the update is the root of the marginal CDF (88)-(89) and the partial
first moment (90)-(91),

$$ w_{t,m} = w_{t-1,m} + d^\star_m,\quad F_m(d^\star_m) = \tfrac12, \qquad
   b_{t,m} = \mathbb{E}_f[d_m|y_{1:t}] - 2G_m(d^\star_m), \qquad b_t = \tfrac1M\sum_m b_{t,m}. $$

`LF (exact)` is $O(M)$ with a large constant, since each coefficient costs a small number of
numerical inversions. It is an exact reference in the sense of Section 2.4, not an online
algorithm.

In [ ]:
import numpy as np
from matplotlib import pyplot as plt
from scipy.integrate import simpson
from scipy.optimize import brentq
from numba_progress import ProgressBar

from filters import (
    # parameter dtypes
    sKF_L_params,
    # signal / environment helpers
    autocorr_matrix_calc, AR_settling_time, laplacian_noise_behavior,
    # regressor window
    shift,
    # algorithms of Sections 4 and 5
    sKF_L_algorithm, sKF_L_exact_algorithm,
)

## Framework

Sections 3 to 5 live in [`filters.py`](filters.py) and are imported above. Section 6 is new, so
it is defined here: the transform-domain pieces (78)-(83), the tilted inversion of Section 6.3,
the truncated transforms (85)-(87), and the filter itself.

### Transform-domain pieces

$\Psi_t$ and every multiplier are rational in $\omega$, so they accept a complex argument
unchanged and the tilted contour $\omega = s + i\gamma$ of Section 6.3 costs nothing extra.

In [ ]:
LF_params = np.dtype([("label", "U20"),   # Unicode string up to 20 characters
                      ("epsilon", "f8"),  # transition noise variance
                      ("b_eta", "f8"),    # Escala de la distribución de Laplace del ruido
                      ("b_0", "f8")       # initial posterior scale
                      ])

_EPS_MACH = 2.2e-16
_MAX_NODES = 2**18 + 1     # ceiling on the quadrature grid
_CONFLUENT_TOL = 1e-2      # |b_prev - b_xi| below this times b_tilde triggers (87)


def q_hat(omega, c, g):
  """(78): transform of the predicted contribution x_n*d_n, scales c = b*|x_n|, g = b_xi*|x_n|."""
  return 1.0/((1.0 + (c*omega)**2)*(1.0 + (g*omega)**2))


def Psi(omega, b_eta, c, g):
  """(80): the full product, 1/(1 + b_eta^2 w^2) * prod_n q_hat_n(w)."""
  out = 1.0/(1.0 + (b_eta*omega)**2)
  for cn, gn in zip(c, g):
    out = out*q_hat(omega, cn, gn)
  return out


def R1(omega, c, g):
  """(83): R^(1) = i q_hat'_m / q_hat_m, the multiplier giving the marginal mean."""
  return -2j*omega*((c**2)/(1.0 + (c*omega)**2) + (g**2)/(1.0 + (g*omega)**2))


def R2(omega, c, g):
  """(83): R^(2) = -q_hat''_m / q_hat_m, the multiplier giving the second moment."""
  acc = 0.0
  for s in (c, g):
    acc = acc + (2*s**2 - 2*(s**4)*omega**2)/(1.0 + (s*omega)**2)**2
  return acc + R1(omega, c, g)**2

### The tilted inversion

$\mathcal{I}_t[R]$ is taken along $\omega = s+i\gamma$ with $\gamma$ from the saddlepoint condition
$K_t'(\gamma) = e_t$ (92)-(93). The tilt is exact for every $\gamma$ in the strip
$|\gamma| < 1/\max_i s_i$ and returns the plain inversion at $\gamma = 0$, so it is a family
containing the real-axis rule and not an alternative to it. Its factor $e^{-\gamma e_t}$ carries
the whole exponentially small magnitude and cancels in every ratio (84), (88) and (90), so it is
never formed.

Two notes on the draft. First, (93) omits the $g_n$ factors that $\Psi_t$ carries, so as printed
it is not the cumulant generating function of the predicted residual; `K_prime` below takes the
CGF of $\Psi_t$ itself. Second, the poles of $\Psi_t$ sit at $\omega = \pm i/s_i$, so moving the
contour to $\gamma$ narrows the widest Lorentzian to $1/\max_i s_i - |\gamma|$; a node spacing
that ignores this leaves about two nodes across the peak once $|e_t|$ is large, which the
consistency checks below make visible.

In [ ]:
def K_prime(gam, scales):
  """K_t'(gam), with K_t(gam) = log Psi_t(i gam) the CGF of the predicted residual (93).

  `scales` holds b_eta together with every c_n and g_n. The draft's (93) omits the
  g_n = b_xi*|x_n| factors that Psi_t of (80) does carry, so as printed it is not the CGF of
  Psi_t; we take the CGF of Psi_t itself.
  """
  return np.sum(2*(scales**2)*gam/(1.0 - (scales*gam)**2))


def saddlepoint(e_t, scales):
  """(103): the gam solving K_t'(gam) = e_t, inside the strip |gam| < 1/max(scales)."""
  if e_t == 0.0:
    return 0.0
  edge = np.sign(e_t)*0.999/np.max(scales)
  if abs(K_prime(edge, scales)) <= abs(e_t):
    return edge                                      # saturated: sit just inside the strip
  return brentq(lambda gam: K_prime(gam, scales) - e_t,
                min(0.0, edge), max(0.0, edge), xtol=1e-15)


def omega_grid(scales, e_t, gam, refine=1.0):
  """Real-axis nodes s for the contour w = s + i*gam.

  Extent: the s at which the integrand has fallen by the machine precision relative to its own
  peak, which the tilt raises. Psi_t decays as prod_i |1 + s_i^2 w^2|^-1 while the widest
  multiplier, T_m/q_hat_m, grows as S^3, so the root taken is of
  sum_i log|1 + s_i^2 (S + i gam)^2| - sum_i log|1 - s_i^2 gam^2| - 3 log S = log(1/eps).
  Solving it beats S = C/min(scale), which explodes when one |x_n| happens to be small while
  the product still decays quickly.

  Spacing: the poles of Psi_t sit at w = +-i/s_i, so on the contour the narrowest Lorentzian is
  of width 1/max(s_i) - |gam|, not 1/max(s_i): the tilt moves towards the nearest singularity
  and sharpens every feature in the same measure. The oscillation exp(i s e_t) is the other
  constraint.
  """
  peak = np.sum(np.log(np.abs(1.0 - (scales*gam)**2)))
  f = lambda S: (np.sum(np.log(np.abs(1.0 + (scales*(S + 1j*gam))**2)))
                 - peak - 3*np.log(S) - np.log(1/_EPS_MACH))
  lo = 1.0/np.max(scales)
  while f(lo) > 0.0:
    lo /= 2.0
  hi = 2*lo
  while f(hi) < 0.0:
    hi *= 2.0
  S = brentq(f, lo, hi)
  rate = 1.0/np.max(scales) - abs(gam)               # slowest decay of the tilted density
  ds = min(rate/10, 2*np.pi/(10*max(abs(e_t), 1.0)))/refine
  n = min(max(int(2*S/ds) | 1, 201), _MAX_NODES)     # odd, for Simpson
  return np.linspace(-S, S, n)


def I_t(psi_vals, R_vals, s_grid, e_t):
  """(81)/(92): I_t[R], up to the factor exp(-gam*e_t) that cancels in every ratio it enters,
  (84), (88) and (90), and which is therefore never formed."""
  return simpson(psi_vals*R_vals*np.exp(1j*s_grid*e_t), x=s_grid).real/(2*np.pi)

### Truncated transforms of the prior

The median and the MAD need the cumulative distribution of the marginal, hence the truncated
transform (85) of the prior on $d_m$. That prior is the convolution (75) of two Laplacians, which
(86) writes as a two-term signed mixture whose weights blow up as $b_{t-1} \to b_\xi$; the draft
replaces it there by the confluent limit (87).

That replacement has to be applied globally and not only inside the truncated transform. $\Psi_t$
and $\hat q_m$ carry the same prior factor, and (88) divides by $\hat q_m$, so a truncated
transform belonging to a different density would leave $F_m(+\infty) \neq 1$. Setting both prior
scales to $b_{t|t-1}/\sqrt2$ does it, since every formula above already takes two scales per
coefficient, and the variance is preserved: $b^2 + b^2 = b_{t|t-1}^2$.

One primitive serves both branches: the six integrals
$\int u^k \mathrm{Lap}(u;0,b)\,e^{-i\nu u}du$ for $k = 0,1,2$, split at the origin because
$|u|$ in (87) changes sign there.

In [ ]:
def lap_partial(nu, d, b):
  """The six integrals int u^k Lap(u;0,b) exp(-i nu u) du for k = 0,1,2, over (-inf, min(d,0)]
  and over [0, max(d,0)]. Everything (85) and its nu-derivative need, in both branches."""
  a = 1.0/b
  al = a - 1j*nu
  be = a + 1j*nu
  D = min(d, 0.0)
  U = max(d, 0.0)
  eD = np.exp(al*D)
  eU = np.exp(-be*U)
  return (eD/(2*b*al),
          eD*(D/al - 1.0/al**2)/(2*b),
          eD*(D**2/al - 2*D/al**2 + 2.0/al**3)/(2*b),
          (1.0 - eU)/(2*b*be),
          (1.0 - eU*(1.0 + be*U))/(2*b*be**2),
          (2.0 - eU*(2.0 + 2*be*U + (be*U)**2))/(2*b*be**3))


def truncated(nu, d, b1, b2, confluent):
  """(85)-(87): T_m(nu; d) and i d/dnu T_m(nu; d) for the prior on d_m.

  Normally the two-term signed mixture (86) of the convolution (75). When the guard has set
  b1 = b2, where the weights A and B of (86) would blow up, the confluent form (87),
  p(u) = 0.5 Lap(u;0,b) (1 + |u|/b); the |u| splits the sign, which is why k = 2 is needed.
  """
  if confluent:
    Jm0, Jm1, Jm2, Jp0, Jp1, Jp2 = lap_partial(nu, d, b1)
    return (0.5*((Jm0 + Jp0) + (-Jm1 + Jp1)/b1),
            0.5*((Jm1 + Jp1) + (-Jm2 + Jp2)/b1))
  A = b1**2/(b1**2 - b2**2)                                                   # (86)
  B = 1.0 - A
  Jm0, Jm1, _, Jp0, Jp1, _ = lap_partial(nu, d, b1)
  Km0, Km1, _, Kp0, Kp1, _ = lap_partial(nu, d, b2)
  return (A*(Jm0 + Jp0) + B*(Km0 + Kp0),
          A*(Jm1 + Jp1) + B*(Km1 + Kp1))


def prior_scales(b_prev, epsilon):
  """The pair of Laplacian scales carried by the predicted marginal (75), and whether the
  confluent guard (87) is in force. b_tilde = sqrt(b_prev^2 + eps/2) is the predicted spread of
  Table 1."""
  b_xi = np.sqrt(epsilon/2)
  b_tilde = np.sqrt(b_prev**2 + b_xi**2)
  if abs(b_prev - b_xi) < _CONFLUENT_TOL*b_tilde:
    b = b_tilde/np.sqrt(2)
    return b, b, b_tilde, True
  return b_prev, b_xi, b_tilde, False

### The filter

`LF_exact_algorithm` follows the signature and the return convention of the other filters in
[`filters.py`](filters.py), so it drops into the same `Algorithms` list. Its `'v'` entry carries
the Laplacian scale $b_t$ and not a variance, the correspondence being $v = 2b^2$ of Table 1.

In [ ]:
def marginal_moments(xtemp, e_t, b_prev, b_eta, epsilon):
  """(84): the marginal mean and variance of every coefficient, three evaluations of (81) each.

  This is the cheapest exact result of Section 6, and by Remark 10 it is also the one that does
  not give a Laplacian filter: matching the mean and the variance is the projection onto the
  Gaussian family, so from t+1 the prior is Gaussian and the filter is that of Section 5. That
  is the closed-form alternative of Section 6.4, and it is what the median and the MAD are here
  to avoid. The mean is needed on its own regardless, by (91).
  """
  b1, b2, _, _ = prior_scales(b_prev, epsilon)
  c = b1*np.abs(xtemp)
  g = b2*np.abs(xtemp)
  scales = np.concatenate(([b_eta], c, g))

  gam = saddlepoint(e_t, scales)
  s_grid = omega_grid(scales, e_t, gam)
  omega = s_grid + 1j*gam
  psi = Psi(omega, b_eta, c, g)
  Z = I_t(psi, 1.0, s_grid, e_t)

  mean = np.array([I_t(psi, R1(omega, c[m], g[m]), s_grid, e_t)/(xtemp[m]*Z)
                   for m in range(len(xtemp))])
  second = np.array([I_t(psi, R2(omega, c[m], g[m]), s_grid, e_t)/(xtemp[m]**2*Z)
                     for m in range(len(xtemp))])
  return mean, second - mean**2


def lf_step(xtemp, e_t, b_prev, b_eta, epsilon, gam=None, refine=1.0):
  """One update of Section 6, for the regressor window xtemp and prediction error e_t.

  Returns the location increments d* of (89), the per-coefficient scales b_{t,m} of (91), the
  marginal means (84) and the normalizing constant Z_t (82). `gam` forces the contour and
  `refine` multiplies the node count, so the checks below can pit the tilted route against the
  plain real axis of (81) and against a finer grid; left alone the contour is the saddlepoint.
  """
  L = len(xtemp)
  b1, b2, b_tilde, confluent = prior_scales(b_prev, epsilon)
  c = b1*np.abs(xtemp)                                                        # (76)
  g = b2*np.abs(xtemp)
  scales = np.concatenate(([b_eta], c, g))

  if gam is None:
    gam = saddlepoint(e_t, scales)
  s_grid = omega_grid(scales, e_t, gam, refine)
  omega = s_grid + 1j*gam
  psi = Psi(omega, b_eta, c, g)                # shared by every m, hence O(M) and not O(M^2)
  Z = I_t(psi, 1.0, s_grid, e_t)               # (82), the same number for all m

  d_star = np.zeros(L)
  b_m = np.zeros(L)
  mean = np.zeros(L)
  for m in range(L):
    q_m = q_hat(omega, c[m], g[m])
    nu = omega*xtemp[m]
    mean[m] = I_t(psi, R1(omega, c[m], g[m]), s_grid, e_t)/(xtemp[m]*Z)       # (84)

    def F(t):                                                                 # (88)
      T, _ = truncated(nu, t, b1, b2, confluent)
      return I_t(psi, T/q_m, s_grid, e_t)/Z

    lo = -max(4*b_tilde, abs(e_t/xtemp[m]) + 4*b_tilde)
    hi = -lo
    for _ in range(20):
      if F(lo) < 0.5 < F(hi):
        break
      lo, hi = 2*lo, 2*hi
    d_star[m] = brentq(lambda t: F(t) - 0.5, lo, hi, xtol=1e-12)              # (89)

    _, dT = truncated(nu, d_star[m], b1, b2, confluent)
    b_m[m] = mean[m] - 2*I_t(psi, dT/q_m, s_grid, e_t)/Z                      # (90), (91)

  return d_star, b_m, mean, Z


def LF_exact_algorithm(N, x, d, w0, parameters):
  """LF (exact), Section 6: Laplacian prior, Laplacian likelihood, the exact marginal reached by
  inverting the transform (81).

  The state is the M locations w and the single Laplacian scale b of the family (8). Returns 'v'
  holding that scale, not a variance, so the Monte Carlo drivers take it unchanged.
  """
  w = w0.copy()
  L = len(w)
  epsilon = parameters["epsilon"]
  b_eta = parameters["b_eta"]
  b = parameters["b_0"]

  # As sKF_integral_algorithm does: the mean (84) divides by x[m], so an entry must not sit on
  # zero. Same 1e-3 floor, and the same caveat the sKF_L_exact docstring records.
  regularization = 1e-3
  x_reg = np.sign(x)*(np.abs(x) + regularization)

  y = np.zeros((N,))
  e = np.zeros((N,))
  xtemp = np.zeros(L)
  w_hist = np.zeros((N, L))
  b_hist = np.zeros((N, L))

  for k in range(0, N):
    xtemp = shift(x_reg[k], xtemp)
    y[k] = w @ xtemp
    e[k] = d[k] - y[k]
    w_hist[k] = w
    b_hist[k] = b

    if k >= L:
      d_star, b_m, _, _ = lf_step(xtemp, e[k], b, b_eta, epsilon)
      w = w + d_star                                                          # (89)
      b = np.mean(b_m)                                                        # (91)

  return {'h': w_hist, 'y': y, 'e': e, 'v': b_hist}

### Monte Carlo driver

`MC_Simulations_Modular_Variance` of [`filters.py`](filters.py) hardcodes `std_behavior`, whose
noise is Gaussian. Section 6 assumes a Laplacian likelihood, so the driver is copied here with
`laplacian_noise_behavior` in its place. Note that its noise argument is a Laplacian *scale* and
not a variance, hence the renaming to `b_v`.

In [ ]:
def MC_Simulations_Laplacian_Variance(N, NR, ho, var_x, b_v, h0, Algorithms, Parameters, AR, PBar = None):
  L = len(h0)
  N_Algorithms = len(Algorithms)
  tau = AR_settling_time(AR)
  measure_init = lambda taps, N_iter: {'h': np.zeros((N_iter, taps)),
                                       'J': np.zeros(N_iter),
                                       'Jex': np.zeros(N_iter),
                                       'var': np.zeros((N_iter, taps))}

  measures = {Parameters[k]["label"]: measure_init(L, N) for k in range(N_Algorithms)}

  for k in range(NR):
    signals = laplacian_noise_behavior(N, ho, var_x, b_v, AR, tau)
    x = signals['x']
    d = signals['d']

    for c in range(N_Algorithms):
      label = Parameters[c]["label"]
      algorithm_signals = Algorithms[c](N, x, d, h0, Parameters[c])
      measures[label]['h'] += algorithm_signals['h']
      measures[label]['J'] += algorithm_signals['e']**2
      measures[label]['Jex'] += (algorithm_signals['e']- signals['v'])**2
      measures[label]['var'] += algorithm_signals['v']

    if not PBar is None:
      PBar.update(1)
    else:
      print(f'Realization {k} out of {NR}')

  for k in range(N_Algorithms):
    label = Parameters[k]["label"]
    measures[label]['h'] /= NR
    measures[label]['J'] /= NR
    measures[label]['Jex'] /= NR
    measures[label]['var'] /= NR

  return measures

### Consistency checks

Section 6 is a reference, so it is worth knowing it is right before reading anything off it.
Three checks, each independent of the ones before it.

#### 1. The truncated transforms close

As $d \to \infty$, $T_m(\nu; d)$ must return $\hat q_m(\omega)$ and $i\partial_\nu T_m$ must
return $i\hat q_m'(\omega)/x_{t,m}$, the chain rule accounting for $\nu = \omega x_{t,m}$. Both
branches of (86)-(87) are checked.

In [ ]:
w_test = np.array([0.3, -1.7, 4.0, 0.05])
xm = 0.8

for name, b1, b2, conf in [("mixture (86) ", 0.40, 0.12, False),
                           ("confluent (87)", 0.31, 0.31, True)]:
  T, dT = truncated(w_test*xm, 1e6, b1, b2, conf)
  q = q_hat(w_test, b1*abs(xm), b2*abs(xm))
  print(f"{name}  |T - q_hat| = {np.max(np.abs(T - q)):.2e}"
        f"   |i dT - i q_hat'/x| = {np.max(np.abs(dT - R1(w_test, b1*abs(xm), b2*abs(xm))*q/xm)):.2e}")

#### 2. The tilt is exact

(92) is exact for every $\gamma$ in the strip, so the tilted route must agree with the plain real
axis wherever the plain axis still works, and must be the one that survives grid refinement where
it does not. The columns below show both: agreement to $10^{-10}$ up to moderate $e_t$, then the
plain axis failing outright while the tilted result stays converged, which is the claim of
Section 6.3. Node counts are printed because the refinement is only meaningful while `_MAX_NODES`
does not bind.

In [ ]:
xtemp_c = np.array([0.9, -1.3, 0.5])
b_prev_c, b_eta_c, eps_c = 0.4, 0.15, 2e-3

b1_c, b2_c, _, _ = prior_scales(b_prev_c, eps_c)
scales_c = np.concatenate(([b_eta_c], b1_c*np.abs(xtemp_c), b2_c*np.abs(xtemp_c)))
rel = lambda a, b: float(np.max(np.abs(a - b)/np.maximum(np.abs(b), 1e-30)))

print(f"{'e_t':>7} {'nodes':>8} {'4x':>8}    plain vs tilt         tilt vs 4x grid")
for e_t in (0.3, 2.0, 8.0, 30.0, 100.0):
  d0, b0, _, _ = lf_step(xtemp_c, e_t, b_prev_c, b_eta_c, eps_c, gam=0.0)
  d1, b1_, _, _ = lf_step(xtemp_c, e_t, b_prev_c, b_eta_c, eps_c)
  d2, b2_, _, _ = lf_step(xtemp_c, e_t, b_prev_c, b_eta_c, eps_c, refine=4.0)

  gam = saddlepoint(e_t, scales_c)
  n1 = len(omega_grid(scales_c, e_t, gam))
  n4 = len(omega_grid(scales_c, e_t, gam, 4.0))
  print(f"{e_t:7.1f} {n1:8d} {n4:8d}    d*={rel(d0,d1):.1e} b={rel(b0,b1_):.1e}"
        f"     d*={rel(d1,d2):.1e} b={rel(b1_,b2_):.1e}")

#### 3. Against a direct evaluation of (27)

The transform route is compared with the density-domain product (27) built by FFT convolution on
a fine grid: the prior as the convolution (75) itself rather than through the mixture (86), the
composite noise (16) by convolving the scaled marginals, and the statistics read off the result.
This shares nothing with the code above except the model. Both the median and MAD of (89) and
(91), which the filter uses, and the moments of (84), which Remark 10 explains it does not, are
checked.

The grid route is the one that fails first, and for the reason (31) gives: on the second row it is
already reading the composite noise at arguments where a double-precision inversion has no correct
digit left.

In [ ]:
from scipy.signal import fftconvolve


def reference_statistics(xtemp, e_t, b_prev, b_eta, epsilon, R=60.0, n=2**20 + 1):
  """Median, MAD, mean and variance of (27), by FFT convolution and no transform anywhere."""
  b1, b2, _, _ = prior_scales(b_prev, epsilon)
  u = np.linspace(-R, R, n)
  h = u[1] - u[0]
  lap = lambda s: np.exp(-np.abs(u)/s)/(2*s)
  conv = lambda a, b: fftconvolve(a, b, mode='same')*h

  prior = conv(lap(b1), lap(b2))                          # the convolution (75), not (86)
  scaled = [np.interp(u/xn, u, prior, left=0, right=0)/abs(xn) for xn in xtemp]

  L = len(xtemp)
  med, mad, mean, var = (np.zeros(L) for _ in range(4))
  for m in range(L):
    f_zeta = lap(b_eta)                                   # the composite noise (16)
    for n_ in range(L):
      if n_ != m:
        f_zeta = conv(f_zeta, scaled[n_])
    phi = prior*np.interp(e_t - xtemp[m]*u, u, f_zeta, left=0, right=0)     # (27)
    cdf = np.concatenate(([0.0], np.cumsum(0.5*(phi[1:] + phi[:-1]))*h))
    cdf /= cdf[-1]
    mass = np.trapezoid(phi, u)
    med[m] = np.interp(0.5, cdf, u)
    mad[m] = np.trapezoid(np.abs(u - med[m])*phi, u)/mass
    mean[m] = np.trapezoid(u*phi, u)/mass
    var[m] = np.trapezoid((u - mean[m])**2*phi, u)/mass
  return med, mad, mean, var


for e_t in (0.3, 2.0):
  d_star, b_m, _, _ = lf_step(xtemp_c, e_t, b_prev_c, b_eta_c, eps_c)
  mu, v = marginal_moments(xtemp_c, e_t, b_prev_c, b_eta_c, eps_c)
  ref = reference_statistics(xtemp_c, e_t, b_prev_c, b_eta_c, eps_c)

  print(f"e_t = {e_t}")
  for name, transform, grid in zip(("median (89)", "MAD    (91)", "mean   (84)", "var    (84)"),
                                   (d_star, b_m, mu, v), ref):
    print(f"   {name}  transform {np.array2string(transform, precision=9)}")
    print(f"                 grid      {np.array2string(grid, precision=9)}"
          f"   max diff {np.max(np.abs(transform - grid)):.2e}")

## Simulations

The setup of the Laplacian-likelihood cells of
[`bayes-adaptive-filters.ipynb`](bayes-adaptive-filters.ipynb). `LF (exact)` is $O(M)$ with a
large constant, so $M$ is kept small; the room of Section 7, at $M = 128$, is a separate exercise.

In [ ]:
L = 3
ho = np.sinc(np.linspace(0, 1.5, L))
ho = ho / np.linalg.norm(ho)          # ground truth
h0 = np.zeros(L)
var_x = 1
var_v = 1e-3
b_v = np.sqrt(var_v/2)                # Laplacian noise scale, v = 2 b^2 of Table 1
# AR = np.array([1.0, 0.0])
AR = np.array([1.0, -0.6, 0.85])
# AR = np.array([1.0, -0.9, 0.95, -0.8, 0.8])

aux = autocorr_matrix_calc(AR, 1, M = len(AR) - 1)
var_w = var_x/aux[0, 0]
Rxx = autocorr_matrix_calc(AR, var_w, M = L)
eig = np.linalg.eigvals(Rxx)
chi = np.max(eig)/np.min(eig)
print(f'Eigenvalue spread: {chi:.4} \n')

### Monte Carlo Simulations

`LF (exact)` against the two filters of the block above it, which differ from it only in the
prior: `sKF-L (exact)` of Section 5 and `sKF-L (min.)` of Section 4. This is the comparison
Section 7.5(v) asks for. The three are given the same spread at $t = 0$ through $v = 2b^2$.
A realization of `LF (exact)` costs about ten seconds at $M = 3$, so the run below takes a
couple of minutes; a single realization leaves the learning curves too noisy to read.

In [ ]:
NR = 10
N = int(200)

epsilon = 0.01
b_0 = 0.5                             # initial posterior scale
var_theta_0 = 2*b_0**2                # the same spread for the Gaussian-prior filters
b_eta = b_v                           # likelihood matched to the noise actually generated

Algorithms = [LF_exact_algorithm, sKF_L_exact_algorithm, sKF_L_algorithm]

LF_parameters = np.void(
    ("LF_exact", epsilon, b_eta, b_0), dtype=LF_params
)

sKF_L_parameters_exact = np.void(
    ("sKF_L_exact", epsilon, b_eta, var_theta_0), dtype=sKF_L_params
)

sKF_L_parameters_minorized = np.void(
    ("sKF_L_minorized", epsilon, b_eta, var_theta_0), dtype=sKF_L_params
)

Alg_Parameters = [LF_parameters, sKF_L_parameters_exact, sKF_L_parameters_minorized]

with ProgressBar(total=NR) as PBar:
    MC_measures = MC_Simulations_Laplacian_Variance(
        N, NR, ho, var_x, b_v, h0, Algorithms, Alg_Parameters, AR, PBar
    )

In [ ]:
h_LF        = MC_measures["LF_exact"]['h']
h_L_exact   = MC_measures["sKF_L_exact"]['h']
h_L_minorized = MC_measures["sKF_L_minorized"]['h']

colors = plt.rcParams['axes.prop_cycle'].by_key()['color']
%config InlineBackend.figure_format = 'svg'

plt.figure(figsize=(15, 10))
for k in range(L):
  c = colors[k % len(colors)]
  plt.plot(h_LF[:, k],          color=c, linestyle='-',  label=f"$w_{k}$ LF exact")
  plt.plot(h_L_minorized[:, k], color=c, linestyle='--', linewidth=1, label=f"$w_{k}$ minorized")
  plt.plot(h_L_exact[:, k],     color=c, linestyle='-.', label=f"$w_{k}$ sKF-L exact")
  plt.axhline(ho[k], color=c, linestyle=':', linewidth=1)

plt.plot([], [], color='k', linestyle=':', linewidth=1, label="true $h_k$")
plt.ylabel("w")
plt.xlabel("Iterations")
plt.title("LF (exact) weights: Laplacian prior vs Gaussian prior")
plt.legend(ncol=L, fontsize=9)
plt.grid(alpha=0.3)
plt.show()

In [ ]:
plt.figure(figsize=(9, 5))
plt.plot(10*np.log10(MC_measures["LF_exact"]["Jex"]),        label="LF exact")
plt.plot(10*np.log10(MC_measures["sKF_L_exact"]["Jex"]),     label="sKF-L exact")
plt.plot(10*np.log10(MC_measures["sKF_L_minorized"]["Jex"]), label="sKF-L minorized")

plt.ylabel("EMSE (dB)")
plt.xlabel("Iterations")
plt.title("Learning curves under Laplacian noise")
plt.legend(fontsize=9)
plt.grid(alpha=0.3)
plt.show()

In [ ]:
# LF carries the Laplacian scale b_t; the Gaussian-prior filters carry a variance v_t.
# Table 1 links them by v = 2 b^2, so they are compared as scales.
b_LF          = MC_measures["LF_exact"]['var'][:, 0]
b_L_exact     = np.sqrt(MC_measures["sKF_L_exact"]['var'][:, 0]/2)
b_L_minorized = np.sqrt(MC_measures["sKF_L_minorized"]['var'][:, 0]/2)

plt.figure(figsize=(9, 5))
plt.plot(b_LF,          linestyle='-',  label="$b_t$ LF exact")
plt.plot(b_L_exact,     linestyle='-.', label="$\\sqrt{v_t/2}$ sKF-L exact")
plt.plot(b_L_minorized, linestyle='--', label="$\\sqrt{v_t/2}$ sKF-L minorized")
plt.axhline(np.sqrt(epsilon/2), color='k', linestyle=':', linewidth=1, label="$b_\\xi$")

plt.yscale("log")
plt.ylabel("posterior scale")
plt.xlabel("Iterations")
plt.title("Posterior spread: median/MAD projection vs mean/variance projection")
plt.legend(fontsize=9)
plt.grid(alpha=0.3)
plt.show()

### What this run settles, and what it does not

The three filters converge to $h_o$ and their learning curves are close, `LF (exact)` sitting
marginally below the two Gaussian-prior filters in steady state and carrying a slightly tighter
posterior scale. That is the expected outcome here rather than a verdict on the prior: the
Laplacian prior of Section 6 is the natural choice when the coefficients are *sparse*, and
$h_o$ above is a normalized sinc of three taps, none of them near zero, so the prior has almost
nothing to exploit. Question 7.5(v), whether the Laplacian prior earns its cost against
`sKF-L (exact)`, needs a sparse $h_o$ and a larger $M$ to be put properly, which the room of
Section 7 supplies.

What the run does settle is that the transform-inversion reference works: the checks above put
the median and the MAD within $10^{-9}$ of an independent density-domain evaluation, and the
tilted inversion stays grid-converged on the outlier steps where the plain real-axis rule
returns nothing usable.